# Security and privacy lab

This notebook is a deterministic simulation for an internal HR/IT support assistant. It uses synthetic fixtures, rule-based heuristics, and frozen injection scores; it makes no provider calls and contains no production data.

In [ ]:
import importlib.util
import json
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

spec = importlib.util.spec_from_file_location("security_lab", Path("security_lab.py"))
security = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = security
spec.loader.exec_module(security)
fixtures = Path("fixtures")
def load(name):
    return json.loads((fixtures / name).read_text(encoding="utf-8"))
sessions = [security.Session.from_dict(item) for item in load("sessions.json")]
capabilities = {item["tool"]: security.Capability.from_dict(item) for item in load("capabilities.json")}
print("Synthetic fixture mode; no live detector or provider is called.")

## 1. Threat register

Threat taxonomies are useful only when each threat maps to a control and an observable test. The table below connects the OWASP/agentic themes to the executable sections in this lab. All evidence is synthetic and deterministic.

| Threat theme | Executable control | Lab section |
| --- | --- | --- |
| Prompt injection | trusted claims and gateway | 3 |
| Excessive agency | capability binding | 5 |
| Sensitive disclosure | PII masking and vault | 4 |
| Supply chain | pinned hashes and regression | 7 |
| Incident evidence | kill switch and bundle | 9 |

In [ ]:
threat_register = [
    ("prompt injection", "claims and gateway", "section 3"),
    ("excessive agency", "capability binding", "section 5"),
    ("sensitive disclosure", "PII masking and vault", "section 4"),
    ("supply chain", "pinned hashes and regression", "section 7"),
    ("incident evidence", "kill switch and bundle", "section 9"),
]
for threat, control, section in threat_register:
    print(f"{threat}: {control} ({section})")

## 2. Authorization is not a guardrail score

A detector score is only a signal; it cannot replace identity, tenant, capability, resource, or argument checks. This run passes a plan check, changes trusted state, and then proves that execution-time authorization blocks the stale plan.

In [ ]:
state = security.AuthorizationState.from_dict(load("state.json"))
admin = sessions[2]
payroll = security.ToolPlan("issue_payroll_adjustment", "payroll", {"employee_id": "EMP-12345", "amount": 12.5}, "bu-north", "nb-payroll-1")
planned = security.plan_check(payroll, admin, state, capabilities)
state.downgrade_role(admin.user_id, "employee")
executed = security.execute(payroll, admin, state, capabilities, detector_score=0.0, threshold=0.8)
print(planned.decision.value, executed.decision.value, executed.reason_codes)

## 3. Prompt injection

Retrieved and tool-output content is data, even when it contains imperative language. The rule-based claim extractor and frozen injection score are signals only; trusted session authority remains unchanged, and a detector miss cannot create a capability.

In [ ]:
content_record = load("contents.json")[2]
content = security.Content.from_dict(content_record)
print("claims:", security.extract_claims(content))
print("authority unchanged:", security.effective_authority(sessions[0], [content]) == sessions[0])
print("frozen injection score:", security.detect_injection(content, {content.content_id: content_record["injection_score"]}))
employee_plan = security.ToolPlan("export_all", "reports", {"format": "csv"}, "bu-north", "nb-export-1")
miss = security.execute(employee_plan, sessions[0], security.AuthorizationState.from_dict(load("state.json")), capabilities, 0.0, 0.8)
print("detector miss still blocked:", miss.decision.value, miss.reason_codes)

## 4. PII and sensitive data

The detector is a documented rule-based heuristic: it finds structured identifiers but misses a free-text name and can mistake an order number for a phone number. The vault is scoped by purpose, role, and time-to-live, while the external boundary receives masked text only.

In [ ]:
now = datetime(2026, 9, 1, tzinfo=timezone.utc)
data_classes = [security.DataClass.from_dict(item) for item in load("data_classes.json")]
print("data classes:", [item.name for item in data_classes])
pii_text = "Contact ana@example.com about EMP-12345."
vault = security.TokenVault("ticket-routing", 2, ["hr_admin"])
entities = security.detect_pii(pii_text)
masked, tokens = vault.tokenize(pii_text, entities, now)
print([(entity.kind, entity.value) for entity in entities])
print("masked:", masked)
try:
    vault.detokenize(next(iter(tokens)), sessions[0], "ticket-routing", now)
except security.ReidentificationDenied:
    print("re-identification denied for employee role")
print("documented miss:", security.detect_pii("Escalate to Priya Nair") == [])
print("documented phone false positive:", [e.kind for e in security.detect_pii("Order 1234567890")])
print("external boundary:", security.external_call_boundary(pii_text, vault, now))

## 5. Tool and agent security

Every tool call binds the initiating user to a role, capability, exact resource, validated arguments, and approved side effect. The gateway repeats that chain at execution time and treats an idempotency replay as safe without repeating the side effect.

In [ ]:
tool_state = security.AuthorizationState.from_dict(load("state.json"))
create = security.ToolPlan("create_ticket", "tickets", {"title": "VPN access"}, "bu-north", "nb-create-1")
first = security.execute(create, sessions[0], tool_state, capabilities, 0.0, 0.8)
replay = security.execute(create, sessions[0], tool_state, capabilities, 0.0, 0.8)
print("first:", first.decision.value, first.side_effect_executed)
print("replay:", replay.decision.value, replay.reason_codes, replay.side_effect_executed)

## 6. Secrets and logs

Conversation traces are redacted and retained separately from append-only security audit events. Redacting each fragment independently is insufficient: the joined trace must also be checked for a secret reconstructed from indirect evidence.

In [ ]:
secret_fixture = load("secrets_samples.json")
print("redactions:", security.redact(" ".join(secret_fixture["samples"]))[1])
print("fragment redaction counts:", [security.redact(item)[1] for item in secret_fixture["fragments"]])
print("joined reconstruction caught:", security.reconstruct_secret(secret_fixture["fragments"]))
audit = security.AuditLog()
audit.append({"event_id": "nb-audit-1", "capability": "read_ticket", "tenant": "bu-north"})
print("audit chain valid:", audit.verify_chain(), "head:", audit.head_hash())

## 7. Model and detector supply chain

Pinned artifact hashes make a dependency change observable before deployment. The same regression cases that once passed are used as a miniature detector-upgrade gate, so a new miss blocks the upgrade.

In [ ]:
manifest = [security.ArtifactManifest.from_dict(item) for item in load("manifest.json")]
mismatches = security.verify_artifacts(manifest, load("actual_hashes.json"))
regressions = load("regression_cases.json")
upgrade = security.detector_upgrade_gate(regressions, {"reg-export-1": 0.9, "reg-tenant-1": 0.9}, {"reg-export-1": 0.4, "reg-tenant-1": 0.9}, 0.6)
print("artifact mismatches:", mismatches)
print("upgrade gate:", upgrade)

## 8. Privacy-preserving evaluation

Evaluation data is explicitly synthetic, role-restricted, and retention-limited. Access is checked against trusted session roles rather than inferred from the dataset contents.

In [ ]:
evalset = security.EvaluationSet.from_dict(load("evalset.json"))
print("synthetic:", evalset.synthetic, "retention_days:", evalset.retention_days)
print("hr admin access:", security.check_eval_access(sessions[2], evalset))
print("employee access:", security.check_eval_access(sessions[0], evalset))

## 9. Incident response

The response playbook disables the affected capability, records credential revocation, preserves policy/detector/artifact and audit evidence, computes the blast radius, and creates a regression case. The bundle has a content hash so later tampering is detectable.

In [ ]:
incident_state = security.AuthorizationState.from_dict(load("state.json"))
incident_plan = security.ToolPlan("export_all", "reports", {"format": "csv"}, "bu-north", "nb-incident-1")
incident_result = security.execute(incident_plan, sessions[2], incident_state, capabilities, 0.0, 0.8)
incident_audit = security.AuditLog()
incident_audit.append(incident_result.audit_event)
incident = security.Incident("nb-incident-1", "export_all", incident_result.audit_event["event_id"])
bundle = security.respond(incident, incident_state, incident_audit, manifest, "policy-7", "detector-2")
print("kill switch:", bundle.kill_switch)
print("blast radius:", bundle.blast_radius)
print("bundle valid:", bundle.verify(), "regression:", bundle.regression_case)

## Exercises

1. Add a capability and test every link in its binding chain, including execution-time revocation.
2. Extend the PII rules with a documented miss and false positive, then update the external-call boundary test.
3. Add a detector regression case and explain why the supply-chain gate blocks the upgrade.